# BioCulture B01 — E. coli 현미경 영상, 분할 및 추적 정답 기초 분석
**교육용 정리본 v2 | 실측 데이터: Dunlop Lab CTC | Python 3.11 · CPU**

이 Notebook은 실측 TIFF/SEG/TRA 정답을 읽고, 영상–마스크 대응 검사, 개체 형태 측정, 20프레임 집계, 계보 파일의 기본 일관성 검사를 수행합니다.

**데이터 출처:** [Dunlop Lab CTC (Zenodo)](https://zenodo.org/records/11237127)  
**필요 파일:** `open-bio-ai/data/bioculture/CTC.zip`  
**주의:** 프레임별 객체 수는 CFU가 아니며, SEG의 라벨을 시간 간 동일 세포 ID라고 가정하지 않습니다. TRA 기반 성장·분열 분석은 B02에서 수행합니다.

## 1. 분석 환경과 데이터 위치
이 셀은 실행 커널과 패키지, 데이터 파일을 확인합니다. 반드시 **Python (BioImage AI)** 커널을 사용하세요. `imagecodecs`는 LZW 압축 TIFF 해제에 필요합니다. Notebook을 프로젝트 루트 또는 `notebooks` 폴더에서 실행해도 경로를 찾아가도록 했습니다.

In [ ]:
# 1. 환경 검사와 공통 경로 정의
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
try:
    import imagecodecs
    codecs_version = imagecodecs.__version__
except ImportError:
    codecs_version = "not installed (LZW TIFF requires imagecodecs)"
from skimage.color import rgb2gray
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries

print("Python:", sys.executable)
print("tifffile:", tifffile.__version__, "| imagecodecs:", codecs_version)

# Jupyter 현재 작업 폴더가 프로젝트 루트이거나 notebooks인 경우 모두 처리
candidates = [
    Path.cwd() / "data" / "bioculture" / "CTC.zip",
    Path.cwd().parent / "data" / "bioculture" / "CTC.zip",
]
ZIP_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if ZIP_PATH is None:
    raise FileNotFoundError("CTC.zip을 프로젝트의 data/bioculture/ 아래에 배치하세요.")
PROJECT_ROOT = ZIP_PATH.parent.parent.parent
RESULTS_DIR = PROJECT_ROOT / "results" / "bioculture"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", ZIP_PATH)
print("Results:", RESULTS_DIR)

## 2. 원본·SEG·TRA 파일 목록 검사
CTC는 이미지 `tNNN.tif`, 분할 마스크 `man_segNNN.tif`, 추적 마스크 `man_trackNNN.tif`, 계보표 `man_track.txt`를 사용합니다. 여기서는 Train 시퀀스 69를 분석하고 각 파일 수를 확인합니다.

In [ ]:
# 2. 시퀀스 내 데이터 목록과 정답 종류 확인
SPLIT, SEQUENCE = "train", "69"
image_prefix = f"CTC/{SPLIT}/{SEQUENCE}/"
seg_prefix = f"CTC/{SPLIT}/{SEQUENCE}_GT/SEG/"
tra_prefix = f"CTC/{SPLIT}/{SEQUENCE}_GT/TRA/"
with ZipFile(ZIP_PATH) as z:
    names = z.namelist()
img_names = sorted(n for n in names if n.startswith(image_prefix) and re.search(r"/t\d+\.tif$", n, re.I))
seg_names = sorted(n for n in names if n.startswith(seg_prefix) and re.search(r"/man_seg\d+\.tif$", n, re.I))
tra_names = sorted(n for n in names if n.startswith(tra_prefix) and n.lower().endswith(".tif"))
track_name = tra_prefix + "man_track.txt"
print("Images:", len(img_names), "| SEG:", len(seg_names), "| TRA masks:", len(tra_names))
print("Track table exists:", track_name in names)
assert img_names and seg_names, "영상 또는 SEG 자료가 없습니다."

## 3. 프레임별 영상–마스크 연결 및 차원 처리
파일의 정렬 순서가 아니라 **프레임 번호**로 짝을 맞춥니다. 실제 CTC 영상은 `(256, 32, 3)` RGB이고 SEG는 `(256, 32)` 2D 레이블입니다. 따라서 비교할 때는 영상의 **공간 크기**만 검사하며, 강도 측정을 위해 별도 회색조로 변환합니다.

In [ ]:
# 3. 프레임 번호로 매칭하고 원본 영상 차원을 검사
def frame_number(path):
    match = re.search(r"/(?:t|man_seg|man_track)(\d+)\.tif$", path, flags=re.I)
    return int(match.group(1)) if match else None

images_by_time = {frame_number(n): n for n in img_names}
seg_by_time = {frame_number(n): n for n in seg_names}
tra_by_time = {frame_number(n): n for n in tra_names}
matched_times = sorted(set(images_by_time) & set(seg_by_time))
print("Paired frames:", len(matched_times))
print("Unmatched image frames:", sorted(set(images_by_time) - set(seg_by_time))[:15])
print("Unmatched mask frames:", sorted(set(seg_by_time) - set(images_by_time))[:15])
assert matched_times, "매칭된 영상–SEG 쌍이 없습니다."

def read_tif(z, name):
    # 압축 ZIP 내부에 있는 TIFF를 메모리에서 직접 읽습니다
    return tifffile.imread(BytesIO(z.read(name)))

def grayscale_for_measurement(img):
    # 기하학적 형태는 SEG에서, 평균 영상 강도는 2D 회색조에서 측정
    if img.ndim == 2:
        return img
    if img.ndim == 3 and img.shape[-1] in (3, 4):
        return rgb2gray(img[..., :3])
    raise ValueError(f"지원하지 않는 영상 차원: {img.shape}")

FRAME = matched_times[0]
with ZipFile(ZIP_PATH) as z:
    image = read_tif(z, images_by_time[FRAME])
    mask = read_tif(z, seg_by_time[FRAME])
image_gray = grayscale_for_measurement(image)
assert image_gray.shape == mask.shape, (image.shape, mask.shape)
assert np.issubdtype(mask.dtype, np.integer), "마스크는 정수형 라벨이어야 합니다."
print("Frame:", FRAME, "| image:", image.shape, image.dtype, "| SEG:", mask.shape, mask.dtype)
print("Foreground labels:", np.count_nonzero(np.unique(mask)))

## 4. 첫 프레임의 원본·레이블·개체 경계 확인
원본 영상과 라벨 이미지, 개체별 경계를 나란히 표시합니다. 특히 미세유체 채널 가장자리에서 잘린 세포와 인접 객체의 경계를 확인합니다.

In [ ]:
# 4. 원본/정답/외곽선 3패널 시각화
boundaries = find_boundaries(mask, mode="outer")
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(image, cmap="gray" if image.ndim == 2 else None)
axes[0].set_title("Original image")
axes[1].imshow(mask, cmap="nipy_spectral", interpolation="nearest")
axes[1].set_title("SEG instance labels")
axes[2].imshow(image_gray, cmap="gray")
axes[2].imshow(np.ma.masked_where(~boundaries, boundaries), cmap="autumn", alpha=0.8)
axes[2].set_title("Instance boundaries")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "ctc_69_frame000.png", dpi=160)
plt.show()

## 5. 개별 세포 형태 측정
`regionprops_table`로 면적·둘레·장축/단축·이심률·Solidity를 계산하고, 영상 강도로 평균 밝기를 측정합니다. RGB 입력은 함수 내부에서 회색조로 전환하므로 반복 분석에서 차원 오류가 발생하지 않습니다. **단위는 픽셀 기반**입니다.

In [ ]:
# 5. 재사용 가능한 프레임 단위 측정 함수
FEATURES = (
    "label", "area", "perimeter", "major_axis_length", "minor_axis_length",
    "eccentricity", "solidity", "centroid", "mean_intensity"
)

def measure_frame(img, labels, frame):
    gray = grayscale_for_measurement(img)
    if labels.ndim != 2 or gray.shape != labels.shape:
        raise ValueError(f"공간 크기 불일치: image={gray.shape}, labels={labels.shape}")
    if not np.issubdtype(labels.dtype, np.integer):
        raise TypeError("레이블 마스크는 정수형이어야 합니다.")
    df = pd.DataFrame(regionprops_table(labels, intensity_image=gray, properties=FEATURES))
    df.insert(0, "frame", int(frame))
    df = df.rename(columns={
        "area": "area_px2", "perimeter": "perimeter_px",
        "major_axis_length": "major_axis_px", "minor_axis_length": "minor_axis_px",
        "mean_intensity": "mean_intensity"
    })
    df["aspect_ratio"] = df["major_axis_px"] / df["minor_axis_px"].replace(0, np.nan)
    df["circularity"] = (
        4 * np.pi * df["area_px2"] /
        df["perimeter_px"].pow(2).replace(0, np.nan)
    )
    return df

single_cells = measure_frame(image, mask, FRAME)
display(single_cells.head(10))
print("Annotated objects in frame:", len(single_cells))
print(single_cells[["area_px2", "major_axis_px", "minor_axis_px"]].describe())

## 6. 처음 20프레임의 형태 통계와 CSV 저장
각 프레임의 공간 크기를 검증하고 객체별 측정값을 하나의 표로 결합합니다. 이 표는 **프레임별 집계**이며 동일 세포의 성장곡선이 아닙니다. 처음 20프레임은 0~95분에 해당합니다(간격 5분).

In [ ]:
# 6. 20프레임 분석: 객체 테이블과 프레임 집계
N_FRAMES = min(20, len(matched_times))
frames_to_analyze = matched_times[:N_FRAMES]
rows = []
with ZipFile(ZIP_PATH) as z:
    for t in frames_to_analyze:
        img = read_tif(z, images_by_time[t])
        seg = read_tif(z, seg_by_time[t])
        # 이미지가 RGB여도 회색조의 공간적 모양과 SEG 모양만 비교
        if grayscale_for_measurement(img).shape != seg.shape:
            raise ValueError(f"Frame {t}: 공간 크기 불일치")
        df = measure_frame(img, seg, t)
        if not df.empty:
            rows.append(df)

if not rows:
    raise RuntimeError("분할된 객체가 있는 프레임이 없습니다.")
cells = pd.concat(rows, ignore_index=True)
summary = cells.groupby("frame", as_index=False).agg(
    object_count=("label", "size"),
    mean_area_px2=("area_px2", "mean"),
    median_major_axis_px=("major_axis_px", "median"),
    mean_intensity=("mean_intensity", "mean")
)
summary["elapsed_minutes"] = (summary["frame"] - frames_to_analyze[0]) * 5
cells.to_csv(RESULTS_DIR / "ctc_69_cells_first20.csv", index=False)
summary.to_csv(RESULTS_DIR / "ctc_69_frame_summary_first20.csv", index=False)
display(summary.head())
print("Analyzed frames:", summary["frame"].nunique(), "| Cell observations:", len(cells))

## 7. 프레임별 분할 객체 수 시각화
세포 수의 변화는 영상 속 관찰된 **분할 객체 수**이며 배양액 총균수나 생균수(CFU)를 뜻하지 않습니다. 세포 분열·유입·소실 또는 주석 특성 때문에 변화할 수 있습니다.

In [ ]:
# 7. 영상에서 확인된 분할 객체 수 그래프
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(summary["elapsed_minutes"], summary["object_count"], marker="o")
ax.set(xlabel="Elapsed time (min)", ylabel="Annotated object count",
       title="Per-frame objects (not CFU)")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "ctc_69_count_first20.png", dpi=160)
plt.show()

## 8. TRA 추적 마스크와 계보표 일관성 확인
Cell Tracking Challenge의 계보 파일은 `cell_id, start, end, parent_id` 네 열을 사용합니다. 첫 프레임의 TRA 마스크에 존재하는 ID와 `start ≤ 0 ≤ end`인 ID를 비교합니다. 이는 **ID 일관성 검사**이지 아직 분열 시간 분석은 아닙니다.

In [ ]:
# 8. TRA 계보표 읽기 및 첫 프레임과의 ID 비교
assert track_name in names, "man_track.txt가 없습니다."
assert FRAME in tra_by_time, f"Frame {FRAME} TRA 마스크가 없습니다."
with ZipFile(ZIP_PATH) as z:
    raw = z.read(track_name).decode("utf-8", errors="replace")
    tracks = pd.read_csv(BytesIO(raw.encode("utf-8")), sep=r"\s+", header=None,
                         names=["cell_id", "start", "end", "parent_id"])
    tra_mask = read_tif(z, tra_by_time[FRAME])
assert tra_mask.shape == mask.shape
frame_ids = set(map(int, np.unique(tra_mask))) - {0}
expected_ids = set(tracks.loc[
    (tracks["start"] <= FRAME) & (tracks["end"] >= FRAME), "cell_id"
].astype(int))
missing_ids = sorted(expected_ids - frame_ids)
unexpected_ids = sorted(frame_ids - expected_ids)
print("Track records:", len(tracks))
print("TRA mask shape:", tra_mask.shape)
print("IDs in first frame:", sorted(frame_ids))
print("Expected IDs:", sorted(expected_ids))
print("Missing IDs:", missing_ids, "| Unexpected IDs:", unexpected_ids)
display(tracks.head(15))
tracks.to_csv(RESULTS_DIR / "ctc_69_track_table.csv", index=False)
assert not missing_ids and not unexpected_ids, "첫 프레임 TRA ID 불일치"

## 9. 결과 해석, 검증 및 다음 단계
**현재 확보한 사실:** RGB 영상·SEG/TRA 마스크를 읽고, 프레임 번호로 정답을 매칭하고, 형태·프레임별 통계와 첫 프레임 추적 ID를 검증할 수 있습니다.

**과학적 주의:** 형태 측정은 단위 환산 전 픽셀 값이며, 단축은 실제 세포 폭과 항상 동일하지 않습니다. 각 프레임의 SEG 레이블을 시간 전체의 동일 세포 ID로 간주하지 않습니다.

**B02 예고:** TRA 라벨의 시간별 면적·길이 추이, 부모–자식 계보, 관찰 종료와 분열 종료 구분, 적절한 성장률 모델 및 불확실성을 분석합니다.

**학생 연습:** (1) 처음 20프레임과 50프레임의 집계 비교 (2) 마스크 가장자리 객체 검토 (3) 세포별 길이 분포와 면적 분포 비교.